In [1]:
# Install folium to the active Jupyter kernel
%pip install -qqq folium

Note: you may need to restart the kernel to use updated packages.


# Mark - Places Names & Geolocation

See that notebook [rendered on NBViewer, here](https://nbviewer.org/github/ronanguilloux/Greek-Scriptures-on-NLP/blob/main/Mark_Places_Names-Geoloc.ipynb).

This notebook uses the `grc_odycy_joint_trf` model to extract proper nouns in the Gospel of Mark. It then uses the `STEPBible-Data/Proper Nouns/places.json` dataset to identify which of those nouns are geographical places and extracts their latitude/longitude coordinates.

Finally, it maps these ancient locations using `folium`.

### Missing Coordinates Strategy
If an ancient place is identified but missing coordinates, there are a few ways to resolve this:
1. **Manual Addition:** Hardcode the known historical coordinates into the dataset or Python script.
2. **Pleiades Gazetteer:** Use an ancient places dataset like Pleiades (https://pleiades.stoa.org/) via API.
3. **Geopy:** Use a modern geocoding library (`pip install geopy`) and try to resolve the name to modern equivalents (e.g., 'Jerusalem'). However, this often fails for strictly ancient ruins.

In [6]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="thinc.shims.pytorch")

import spacy
import collections
import re
import json
import unicodedata
import folium

# Load the transformer model for Ancient Greek
nlp = spacy.load("grc_odycy_joint_trf")

# Read the Mark Greek text
with open("data/mark-greek.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Remove verse numbers to fix tokenization
text = re.sub(r'\d+', ' ', text)

doc = nlp(text)

propn_freq = collections.Counter()

for t in doc:
    # odycy model often tags Proper Nouns as NOUN, so we also check for Title Case
    if t.pos_ in ["PROPN", "NOUN"] and t.text and t.text[0].isupper():
        propn_freq[t.lemma_] += 1
    elif t.pos_ == "PROPN":
        propn_freq[t.lemma_] += 1

sorted_freq = propn_freq.most_common()

def remove_accents(text):
    text = text.replace("(", "").replace(")", "").replace("2", "")
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn').lower()

# Load Lexicon to map Greek to English
lexicon_path = "STEPBible-Data/Lexicons/TBESG - Translators Brief lexicon of Extended Strongs for Greek - STEPBible.org CC BY.txt"
greek_to_english = {}
with open(lexicon_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("G") and "\t" in line:
            parts = line.split("\t")
            if len(parts) >= 7:
                greek_words = parts[3].strip().split(",")
                english_gloss = parts[6].strip()
                for gw in greek_words:
                    clean_gw = remove_accents(gw.strip())
                    if clean_gw not in greek_to_english:
                        greek_to_english[clean_gw] = english_gloss

# Manual corrections/additions from the person script
greek_to_english["βηθσαιδαν"] = "Bethsaida"
greek_to_english["ιεροσολυμων"] = "Jerusalem"
greek_to_english["γαλιλαιας"] = "Galilee"


In [7]:
# Load the places data to filter locations and get coordinates
places_path = "STEPBible-Data/Proper Nouns/places.json"
with open(places_path, "r", encoding="utf-8") as f:
    # Skip the comment header to parse JSON
    json_text = ""
    for line in f:
        if not line.startswith("//"):
            json_text += line
            
places_data = json.loads(json_text)

# Create a lookup dict from English name to coordinates
english_place_to_coords = {}
for key, place_info in places_data.items():
    if place_info.get("type") == "Place":
        for name in place_info.get("allNames", []):
            coords = place_info.get("coords", "")
            if coords:
                try:
                    lat, lon = map(float, coords.split(","))
                    english_place_to_coords[name] = (lat, lon)
                except ValueError:
                    pass

english_place_to_coords["Galilee"] = (32.76, 35.53) # Example fallback


## Places in Mk, by mentions frequency

In [8]:
mark_places = []

print("Places Found in Mark:\n")
for name, freq in sorted_freq:
    norm_name = remove_accents(name)
    english_name = greek_to_english.get(norm_name, "Unknown")
    
    # Basic string cleaning to match the Places JSON
    clean_english_name = english_name.replace("?", "").strip()
    
    # Check if this proper noun exists in our Places coordinates list
    if clean_english_name in english_place_to_coords:
        coords = english_place_to_coords[clean_english_name]
        mark_places.append({
            "greek_lemma": name,
            "english_name": clean_english_name,
            "frequency": freq,
            "coords": coords
        })
        print(f"{name} ({clean_english_name}): Frequency {freq} | Coords: {coords}")
    else:
        # Might be a person, or a place without coordinates
        pass


Places Found in Mark:

γαλιλαία (Galilee): Frequency 12 | Coords: (32.76, 35.53)
ἱεροσόλυμα (Jerusalem): Frequency 10 | Coords: (31.777444, 35.234935)
ἰορδάνης (Jordan): Frequency 4 | Coords: (32.309099, 35.5599)
ναζαρηνός (Nazareth): Frequency 4 | Coords: (32.70674542474383, 35.30152807767973)
βηθανία (Bethany): Frequency 4 | Coords: (31.8363211491438, 35.55260017422344)
μαγδαληνή (Magdalene): Frequency 4 | Coords: (32.84733494629723, 35.5229361797708)
ἰουδαία (Judea): Frequency 3 | Coords: (31.777444, 35.234935)
τύρος (Tyre): Frequency 3 | Coords: (33.27582782266882, 35.19257453545583)
σιδών (Sidon): Frequency 2 | Coords: (33.56316734135746, 35.36634649354799)
βηθσαϊδά(ν) (Bethsaida): Frequency 2 | Coords: (32.90784828392077, 35.62697295608815)
Ἱεροσολυμίτης (Jerusalem): Frequency 1 | Coords: (31.777444, 35.234935)
ναζαρά (Nazareth): Frequency 1 | Coords: (32.70674542474383, 35.30152807767973)
ἰδουμαία (Idumea): Frequency 1 | Coords: (30.734691, 35.60625)
Δεκάπολις (Decapolis): Frequ

## Dynamic map rendering

Map of all named locations mentioned in Mark, by occurences.
JS-generated, using OpenStreetmap+Leaflet.

In [9]:
# Ensure you have folium installed: pip install folium
if mark_places:
    # Calculate center of the map based on points
    avg_lat = sum(p["coords"][0] for p in mark_places) / len(mark_places)
    avg_lon = sum(p["coords"][1] for p in mark_places) / len(mark_places)
    
    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=8, tiles="OpenStreetMap")
    
    for place in mark_places:
        folium.Marker(
            location=place["coords"],
            popup=f"<b>{place['english_name']}</b><br>Greek: {place['greek_lemma']}<br>Mentions: {place['frequency']}",
            icon=folium.Icon(color="blue", icon="info-sign")
        ).add_to(m)

    # To display in Jupyter Notebook, just place the variable `m` at the end of the cell
    display(m)
else:
    print("No valid places with coordinates were found to map.")

## Static map
Same map as above, but PNG-rendered: Map of all named locations mentioned in Mark, by occurences:

![Map](assets/Mark_Places_Names-Geoloc/map.png)